# Satu uji referensi kit: chunk 1 → pembanding chunk 0

Target: karakter kit ringan chunk 1; kick yang disukai berasal dari 0/2. Uji ini **hanya** menguji pengaruh referensi chunk 1 pada chunk 0. Tidak mengganti kick, tidak mengubah caption/seed/CFG/steps/precision, tidak mengulang seluruh lagu, dan tidak menjanjikan timbre terkunci.

Gunakan VM yang masih memiliki runtime/bobot ACE-Step. Salin sel ke notebook aktif agar cache dipakai kembali. Helper sudah tersemat. Hasil baseline tidak ditimpa.

Jika file chunk masih di VM, hasil notebook 15 dipakai langsung. Jika hilang, upload **drum_chunked_result.zip** (paket WAV, bukan feedback.zip) dan piano asli. Referensi memakai WAV lossless chunk 1, bukan MP3. Potongan 30 detik di tengah core chunk 1 dipakai sebagai referensi; piano tetap satu-satunya source yang menentukan konteks lagu.

DRY_RUN=True menyiapkan input/plan tanpa model. Tidak ada pemasangan runtime atau unduhan model di notebook ini. VM kosong perlu setup source/runtime/bobot notebook 12 dahulu, berhenti sebelum generasi bass.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid
from google.colab import files
DRY_RUN = False
USE_SAVED_ZIP = False  # True untuk memilih paket result, walau run lama masih ditemukan
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU pada VM dengan runtime/bobot yang siap"
for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel memerlukan {dependency}"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "electric_full_song.py": {
    "sha256": "04d99be5e6bf1635b7ac16347680f344be13344fa1b8953523c9543a96c502ce",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwXVyr58lJikRTMGpurAjc61gtkC3/GWE65CK9ojEGYB5lkohzY6v379aqRDJ5jxyVVkpV/4bJiI4FwjJy6wyViLFFjktsrBqu7AHgk4+FVaMIIOUKVoefIkjB25dg6R4lX19z6F+S8EUXs/Ut7ktLYsO95MK0478LfP22Q3YZ3B8jJY9XudOqgJXsFG5WlCpzDOZ7eL5tlDp32j7JNZKSTXeXfuv/v8i396cvqOVB58/+vZp/r7w/X7X05O/3Z2+VZ/nV5dnl/8+vbio2PZ72nh6c1ucWmMdFqbc9tBc6C3zjBE7+oDXJ2fv7+4PJs6x7DPp48nlzfnVx8/nH28sXu0vmvA0N0T1cYjB5hwaYRiPcCuzEvU+na4tW++1rAm857m4dI2EanBw+QOBrorN+K5aPfp57cnYp3Ubgcm8yjyrTFHyu/0+vPow/WNmAdxPAvCe6dxFp0x8cCKi2KgWIz0Aj/rT3skzoWrFXYb7IMzmKsPtK+Bme1zTsK/42Vca0nvJJQIF91lXRElkk+Z9RVIO58FLVKyX6d43ELGc5BNQb4A4Xt4eP+Av7oZLm5nDtb1AsS9XDIw6QWnXYGiRYn2hAqBoHaa8Qk8iRdh7QRQKWNQlx40hmU5Htu5vTY1CnfC3uit+jRCP3v028nZtC5dmMfrYml5Ga3YdrHOKM/6LLjagGoCG8DjbwLp0y00aBlaWpSb/brX241RgEYT+824iCYjGGUICuf/wr8tCjKVP7rHbS/QJhwfgwq+/yICqcBnJu0EU+PobOZTAhlQz18+gq4xnAKL7BDsPwX+4Y7bqm4EJR7TntXWmQUE3lK9/an6vp1aE2lHEl/N9m4sTpNCCxr7QdcGFN6OvnQFRYcAYEAL8TTyL+IUpBf6JdpkccEj26QKfERTOvWdXbDFfgDVax2xqca5FU9rORRXnp0Hgn07aSFjZpD93nA2TSNbIgud00+bIZa6uT1Mx236IjVg7shcaf2rz5nFQeKaLbUmss2xIVYytacbtYjpjd/TmTFYMcFheNyvbJCpqaRCriZBMjVEyNVYOzww4I1GLZQOQTXLqaaYMINVpk5Bcs9Hued0XCYLigA86yLanq1btKUExAYwapWruUadT/erc016Uw3MWmKysVvddjdABNfygPBN1BygWG24p9ZHy1YN7YMWEDX0gIKmNTCGFQVOaxLoQ2q1Whv6WMqGomSdg6CnOE4+tXw67tH4KPNtLT00+RmOYz5vowEjQlmJ5ZnyjH4iAtEkQJ1qrdUhTHCAynUxdeZgvAAnDXnGdO48kjKiL5TQaPP5/hN4stTUKPJBJc62KEqk2Dr7NxKWcYvQuGE52MkNh+kaWY6Swcd2SuhFSZ7/jxxilQ8MQIyupuigsm88faTDPw1bicL94VJr3T+cHFTzdloQbDA6xD7n6YqpMwI/nEpAkPPyjdSFvEFYVkqIV+cVHYM+kAP3wUJqx/gPowj9mibSycNxq0pVP8bKxoSiLsO6gNWHc26UfPBW2WtEoeus1BcqBTEj8bsxyDII/g8Io2LyvVnmBmZjNYODxQBTWmnGrcf/9Oq+veZunIOUAriibOZfVLhTuE2nUReEaMhW9TKgkop1XHpfVeYgBCvYopPeAXgTDw1oD/bc1t5vLmWEuqC7o42oXtxZyKMghq6E9v5NZedYqlSBiu/rfKGgIQeBMctYjb+49t+enb8HxnvLlTZga6tNy4ybU21qDHJswx73oYYz2rpksE3MRA751vubacPql+GUJoImdqTItbV1VatqVNCoEWqyTJ1RR742bmGPLBgOfYuMx2P/hHDuY7H2eKTJwgzVSmuk/9axSFPdRAqlwmxTDDiO8zcpMxGIYgUaQ2DUGXUvuc8MDhHMMa9xdfVhhEvB+vWzgFVQUCE/VumEwRo0exWOaickdtBYo7YNgwhkBRbuc9SEY/8UIjIhmz7ctys4GyRsFU7uJDUmdRqtsaGxKnVwsV1pcbir2GJoLH1tHWvi7FkEs0ZB5oLQmmGl30RXJltFyMDXfRMHfyzkdnr14cPFp5vh3kqPb4i5sWYio9XXKsCaAp7IYiGjxg5VEBAL0iO0BG2q0jkBC1msYZGMLtPyHHRlZAprdOoZp82xHawsnm7MLLZ6aSWr8NOYuI23G8yFVhG5mWiVnWMq0qp7sAdhxsvHSIiuC+Sw7j4zofE2yERsIjWfY9U9P2wBg6GgQ4IDZxIutSLSOjbiCOafXaM/ade+mwih2XbQxZF9PSp+Nynfxhsm/Z5BPz/iWHqab7+tNkALPcr3kUmOeSWW87CXeZBDySbLfWNdgHHrCT1W8iKQqvjDZbx11K9jHDMfM+kK52mGuiUfDDN+LY+qk/B51sPq7spRMXyOVdTurs65YQUv3gufcmGlpkrW6ZoicvWTLkoeRjIDIxpZuX7OhdRgXnKJ307OqHCkVBSEBfVB7ogn3gV59AAoAAelJOI372dgg4Ksl3CLT8JAEyUS8/xcIldKQrTn7En29KhnRmWPJ2el5/pL3hpBjaFoFY7apbW4Q9uFwwBFQhPJLJg2U0/fegOLc7QqmPQEVufO9fuTywm9vXs0G99+10zbfXf3VBicUkz81+vPonrd9Xy4lY910N756vJMXJ/c3Ezsnes9J97r+ZMunB4+czgdtmcKrV8OVjS244xW0Q5a593KfV0KTirRJ2Hj+5ZmJIfiiNN/ZPEf4SynTmjwAMBZw/4f1CFJIhUKj7Hgk/ncJ3cMA6w13VSKrC6kAT043Zlh6roVNfndOnxgE/6gEGbP7qYCTWvPTnjNUkRuX9iuY8sOGkrKJlJrJKUctCTllIMxgSIHc4v4gqQefuuA9i6wXA5GHr+4/j0lkYU+PVXDW/kEJiSVbNKQY4/1OWEnDCp29r4d31nPmui5Un8cs2OXOxZm0ECnKKqxSapXUfB7SIHpBsj0DJOsJXVMRv6OnA2OgJVuW9UAdziiYq1253MAtTMx+kC8Mzs4aDOouWJIasfCAiiHsqkUBPQROby4BJxCliUVptzdOjwI9VtfL0+1QvzSoBgVcv2alLWqlRPQ71ixJmDY94B11/PV/koau76hgmWr9AG3LMH/oUA+6s1YLtLmK9sgbryG7WwDBjX4WT4/F1VfzZZkjeDqIOnQvbKawMUChi79GhLc07++GY13KHpWwQc462AhTdeTTZD4AIKRRSR4L7dEffR2gcxFaBnqTzAYDaI8he/63MGzj1dOWLc3HobQfsDDeaEN1x67le2skSm8ajARk4+WOfSsmc6MD7zw/ZrVO8PKA79QXzVTHbcHIIxyEMzpingbRumUEkHquct9TirCteKtuNCIdhYV0VvKi733nZK438vvl8Fk4aEaxB+ELfpBLxR5Nl2ShgHN3oLARfDBpwEP5QPg1vgsDTQoJjwxN2qxJnrEjdUaUl73sJBvDNN81JFo7ecwfZOI13hOIwfrmFJ3ulF7JSTlUAvSvSuEPRdhmwcvQmBkGMImg5c/z6pVE9bwsE4a6dyDaGYcDGCbdfJNod4uKLvrPCRqV6pVcSqrQG6wV1O8fMlG6XRr0UZh292zALsgJYPP5Bb8ugW9O1ubWznEga0i6vRE9b640i7opKKhhwK7Pn5fJLU+d6NM7m7HVvsX6+ltxGd3bVa/q3vuhhiD7WzWirE/f6V+F6a20YaVqJp2FTam8IiKfU3FU/33rmizpYN1rs62cjq9zs6wdSaDe37+DbyIth0YWbMtOIqNBXeOesnCOs2yf93moN3LEhkyI0w7vNSlHzD4EZO7JjRj+7v2ZHrDQkzrqSbMLHOQ7qbtzdCQF9PG9HmS+VbvtZWIOtjtVb4wi9pPu/+LDKoOcO+y99sOkC2x9dRGDKvpnvEneWeWgjfzetX4t0Iaox27Av+WVCEgVwDGMl2QseIv4iOQgOIXN1VmseLNEUt+tmikmez9wSz0i66j39/DWUxEnkpb0B8zZS7eSb4gW/uaetxIcgQS4+a+H6Uh+PfWTPwnHfxAT3Fr/x4pZHpNOQXYMgBGn+6KEeh//KK4Pb7bu3L1T15Ua+8dzjEF6xwmLGr5/Lt2ohpqPZnMmfoW470zrZdv9mT896KmztVKla1nfyZKtgq+qNV6ha92jAW7YwcdqqJaGkIKKOQU87hwqf0zqyhMz0R9xIoIbj5fo0d40yiF0evSX7iyybmib4PFkLyBXZPRqP6mMUgeduKEGgFV7X8RRDZ8ArNDUb1k48dc+kSSrR1DIthvkO20FTPco0oZVSca2svXhzK5I+5sxiy5sS6kN9kkatYfg1ZEz7khiTqpOPLWWMbo+/6H4LoE7DYnQJZVWHzIMpacB5/rMX30pYmTD/4HNKQt7d5MAAA="
  },
  "drum_probe.py": {
    "sha256": "e1197fddd057474b7cc0b9258bb0851a15996fd9e2d96281a99bd4fdf65dc50c",
    "gzip_base64": "H4sIAAAAAAAC/7Va3XPbNhJ/11+B5B5AthRlx9NmTj4+uI1z07lem0vSduY0Gg5MghJOFMkjQNlOxv/77eKD30rSds4PsQkuFov9/O0yz58/f1U3R1LV5R2XpKxJ1uT5UpbFjkhR7HK+rJiUpMEHovacVKIoeEpuvr9dvlO8It//8uqG1E2hxJGHz58/X4hjVdaKsHpXsVpy9/wf4LnI6hLOYmqfiztiX7yBR0ck940SefvU3IFcCZfSrXwQVSZyvnDPPOeJqkUSo9SxlppJK/hi8ebt7bvb9yQiHxcEfqhI6ZqypGykEskyhXvL5emSBvRYpjzX77iEO8HiN8s7Jjm8SvlJJBzeJU3KaGAYVTVPhBRlAetZXjJ19QJIRSEV8OSFgmXNHRYl56mk683F1u7dNSJlRcJjmbAcGL8ML3BrxmuuV+F8oP8GFssMeaexKuOkauj6fd1wyyRhlTLH/yDLnCkwiLsXwZPJQYAFkqQ8VqwQKFJIwEpwu5oJMF9ACqaamuVkV5fliZN7ofawKTkERBas5oQVKUnyUgLnvVjumQpIVuZ5ed/5AStKUvE6K+sjXoiAB8DLgFT7mml3QR7pY8GOIpEh+T7nrDbS7YWSAVEMLguGI2DSXGpqWYEJyB1X95wXhhGHreii4JxF/giCl+RUguq0sx55XqZw5071MgQjPS0Wi5Rn4GnJge24VzaqauAGsknQm/y11iIIxiNtqDgDK90BcQgORvVLrRDrbuG/RfUafls+K9wY0HsaoH5BqegJUUv7w5v41e3rH2/e377y0RtZnezFiZsz8QcUBqYAq8EqSE42RgYdgiGGCbgNsIXo4elKFHCiW93xgtcMLb/iD6B6Y1nzsmU/+elvk0baGYYXFxfxTazdNkanvVjBzZr8C9ijMzY1X9nfjrdhBRc5CX4fHqsrDDTxMFjZrgd8RUacjp2C/FDIGBXr+WuryfC+FoqPCYN2x8KystbujviUTY0N7JXRC/6Egb/UyFpD4T07Gc3ov0YawZ8vvbbxeWtS7kFgq0CWTZ3wwAbAV4FUrFbRRZA2xuzR1UXAH3jSKB69hqBCfi4VsSYVZfRTWXAbMFOO0Qazt1f5IWipzE9gJX3zCq88I4G/1YxMgg6ZUizZazJjNFMeakhIhc4obVVgL775NtD3j3UdaYlt4Yl5cRJ1WWA8uF0zr7p9kC440+ZAF+ltifNyt+NpR3lkSXxf1gfIxfd9OnANSHs9Qp1sax5D4ZCQs4tWfJusNRdeg4L/2wigy6qrF73dUF7itq64ra9//Pnm/dWL+Lef3/7j9u3g2h33ZM/T+D/lXVDlzKgmKYtM7CJU7IqaB7kaFj7jgy5UDE0XajoZI0WIFUh69j3oDMoRf1Ce7z+LTHnt/BVKi+TkV5Y3/Lauy9qjFlpwyRVJRQaOJa/Ncw1RgVWkzFPSpTJ9bANMGCn4vd25OsE2LHbGR4wjhcdDKmoPFQ45P9KlkT8IqeLyYBzZUEMaBSVFHyk4vmqgsNrMCg54rmKbU+naXG+Y+DRSAFrjwDFCIwcfYqhJuYzRuggiAGpcjvYWpcK9P4NngD6VKBowiCmI+AxqvcbqplFMsm+KA9ZQCCYwmoIMUOxC8uvNLTjkI5TaHLOK4nUBpz6G5KeSoN/BfbuUjns1sIA6ApUSiFffvfknVHUmwP70ydS5NqiGmbCfqIwSjUJV/dhZHJiylCkW2YC2VcsbxPyqLWbUph/9b5eC3B/+olcHRmmICAm6UQSz0TBDjvOVTkijxV56Guz9C3ldc/6Bd0zI3aPiUtuhQFyTiw9atoDc/isAMoNxULM1+CrGbV1W4TDz1+UHXkT28h5tedOvR2KFsskyyPuQWXgNITVgU9ZiJ8C64FFZvGdyH5k0OLnbYJcB0GFSVo86kkfEgRHOH1ddy9q+fRZNDl9Pg/ttq7NkzwrImQQMicrBw6n/SSOZg0Yk6GQbOiKlWwhe7BjoGmLVSQi4WktM1xNJnxZDjmFTpVgOTQKIKCbJAn1RI6vIuXAn7++NCKtCx2hj04Hzarr9GxRZndlmfHrqz7auRL0aYwqpzUfToHIrXeTTtqbb3/7cEV9HG7pcHtnDEorHUnJIQimms6sL+Ge5dI3K0jQqAX2ply3KW6ZAn/IaY7puFWVfUlvne+qxgkzRjU0dRmnVo9qD3jtEMKeZjf0Deyq6TAClDQqsNyyaVm+btmfa+sFXjsPlejuUtEMAnqUJAD9EM1jCCBlA8U0Ouv50AvP83IXbK6DSzt7c1zrViSzWBTmOexnMRyuYq56D5UCAjIxtjJywBEbqW8v/v9xcjm7cOWU046fX3d+2og/N3kGDCOPWmzo/9uwOStMAXOIcA5cHwN8RLMXW36NvLwAN6ya/q92R7faBd6zNEzknMpOC7SfaIZO8B627gSfDxj96GSCvaAAlhrL3ElGvT6OTzq9bGCQkLJfOCaFwNXVhM+JiRrs9rOdN7TTpNv0BFFwM4ryl3HHlOdwFNcXxA6th/cx54XXEGwo2kRCbz6LLUX2fFJ5b2JXgyKPUSOpY5RyfNJACLj0twlM0PQOc5DoD4MnrGZec9sG0V1CwO4w6xO0ZNgGCb90nDREMUrskryGD6S61XuxcCfWifQ3fAxvzsodLkcA4t6XYUIOuYlsBt8+irvSMXn1OkxqfG0nMoMH2/AQgt8hEYmBkxiAFpT21Quixowxcl6G3wNlcAabdoYKpoYA4mXtp9tHtwA/RoFDp28HWJGubURpdXwTTEdr8AG0mRmlbldfn63VAFZOHWD1WCNdzvisHrQLL+6O4EX9Dltjh3N+tx+tWR1sR8DNLDojVtf/qFwYM2BZgPZvTKTTEd5AlBpiUrk3HThU0B9gstAvQQgF2UHGrTbM+x9iRojrkhAHk3F3Ddty+ePpDCN2Zdw7b6QJ1DtDCIax49Iwz6cg4QDic9IThEJxwxuB4h5AqjxJy0QxQvUlUw3KTH5wf2naUDg6zXa7JW3puC8dBtsDI67+7Y9CPxVJ80PF7OX6NyjPdgU4g1Heq0Tr8XEz+Urg7OYl5utIntsL3pLaAazZ/z87lhpn7uj+M8CzhQCe2MzCJxe/nmhZ5u2yje1VDPq7TK7NfT7j8P5GwfsAkNe43tJp6oOJL+0gzcZ5vqp5FZ/uRDf1Sac92SLPCWjgI1odzzWG2vqzaLApOrgEhJKl+b2KHiJ1aT5Ak0jiDuOFYVL/kjHYC2XJuFz7PFyNS1kmQShXsGMQlTjy9wYBzZhwcvvQDrztmMh0OLv2RjrsvQiEAU29Ds+xYcRzlLItSqlTgWHWJvDiaQK/jP2LQpoCkCIhZRoOMnsoc0nr0EcV+op8GdtBtrAEW0lzcgXw5aAGZ3+m1y7++OAxOAV3421mUPN+YdvAocIE+U1sD23fFLvLt73OCa5cxDhuNgrOzjh+g6uepnHn8cyf0Sq+F1H2cM3l7nlHF2SHWc1CW52WCqoj1OGbA8CwV9f9QD389+kg0NtUYOPOHhFeKfAcV/Fb/iRCJSaIdbv0ZG1soFWjiKKMfEWR4+skP4xg/LMXx05p81EtP9M9NJXBWN7pdbzTau0x7kTW2IcOBrhk6Cz1nXP8eKRYLSL/uTlFE4/gIIRbH1CipitwX4vCm3mlk9Qafai/lMqmFlieK47RMoP01W0KWpjGz1F7b5aIS9dwPgiNjCNHP9c6hHRhvLrfnWJqK1GMKHjIh0q10d6ytomkvznd12VSR2XlsEIDkjzGoO2+kOPFYv/Z6pJMjWjxqztGfmdv7XV18amv7CR87WK1GqBwlVHmAppyeu7gbKcxsCfY879nr3S9vEJ6/O2sVV/eWpmT2dHlO6TgRnr3pxZz6bVv7iesxUL0WFvdJr/0myEJzzX6gWmH1l65R/MY4eYxYCPbunGjq80PA0W7sfbqcGXX2MVt/02xnPWBuEuEYmPjT2Zr9RIRjUShZ09dQG87jnOtOMzqckHpjRrHbcwP13uih3Y1acB8pY1V6I236CButhOEYjRll6PQOlOGeP6RixyWo5llk5JnHYucx2W/a/L2BP15oheeTo5BHxNlzmEz7mxswshDzTneFYHSlwHxCskOjoGsdXSs/+cjaPpujh2M07bYmLbUj4yrk5j4uYeEbl4b6rTrgTeVpT0ubYyW99vOwuUTgOJs72A80sDj8RKPhspaj+482KCQsOJp25Mycuia3hCNHMBt0k6JeXoCr/w/cELTpiyQAAA=="
  },
  "ace_context_chunks.py": {
    "sha256": "8ab7629c197c5f557ba20b1e978791db23fe5ef16044f76a5eac0e227b9610c2",
    "gzip_base64": "H4sIAAAAAAAC/60825LbRnbv8xWQ9gHADAhyKEtrcwaq8irazVYSW2U7qVQxLFQTaJKIQIAGwLlYUVX+IX+YL8m5NdAAQUouRw8jsi+n+5w+9z7Nly9ffv/u/eTnRh+cdXksUp1OkrJo9FPj6KeDrrK9LpraUUXq7LWqj5VOA6dW+0OuJyrPtoVOnbrJmmSXFdvw6uonfax17TQ77SQ7nXyE7rWq62laHffOY1l91FUdOh9yVUwPVbnWTqo36pg3TlM6RelkxUZXukh0ePWD1mntFMf94RlWTDL6D7e4yXIdOJvN/qC3gXPIj7ycfspgI8XWefev//C9Ux2LBvbubMrKmUz0k06ODQB9+fLlVbY/lFXjqGp7UFWtzfedqnd5tjZf/7MuC/N5r5rd1aYq984BPsEgRzo+YId8rnfHJsvbb8c14Jfour4yTYSKowCnQzvK4IPN9YbXIFzDGmircrNQUlaVzlWDiGdFGh+0+thB1rlOmipL4s0xz2PY+JbgATFyDUfy/S/vnch5NZ/NZlc//fjjL/AFN+7FMS4dx35Y6brMH7Tnh0ASPPDl7erq6grOBtBS89dvPETcX1w58C/NtrpuIqFXKAN86nvMmh0DpwlhedCF51Zr16cdNZVWe4aC//Bw1nmZfIRzd7JGV16u9utULXggbEul3u1s/s01/vGDtev6C14+PB5SIIdH03ntSjfHqpDthTv9xJ9gZ4zJYwVLxHiutLngQeVHLThhQ9Tt+67RSFlVPUf4PUS04vq42WRPnhs2+4PLS7bDQgaOcuPhCmEKh117tEQABwY0jeaByvPyMS5UEf1V5bX2b9z/KE4gVfqQq0TzRmTruXouj43XlI3KA0cENPpuFgBrVtusiN4AX6hUR+Hrb6H/qOvoh7IAZtGFrrbP9EUwpS3VkSdAAgYQ4GzeSbZxPFU8exnwT90okEXvIViXZe47cFxF2Ti9Hi8rmmCTl6rx2wEoL2FWA6cCUbwHnw76AQ+ZV/dbDpAJt7P7SDZ0H72ZzUz77B73dc97xEb+dH+Ln2XCZH5tWmd+x1uVymrt/Buu976qyspzUaGYSbDi//73/8BKtSHh2+i2ZiLiPr2ZEMZHclcaeme12xJoQAU+FyCE3yF0jWJ3H1EX4UQNl/b3c3msEu3sj3XjgGI0O3RqDZtOzep/cv4OGGw10KJMde6kx0o1WVnUsO0kP8L2Nejqg0pTVMjOD/oBhuqnBJFHNVnpX+EEGiSFOoQEcq+esv1xH9GxwVHCZoRMPm1aJCSNKlRXHhOGevw7ok4kEOAoYJghEvbc334Zb4OPORrYl5Nr9QBmJAf9CbtWBVCjdkqG6ahjmpVCjgS21ES3uJ6QWvYCShHWgS/ePCDEEp3lHh8V79OfIjRf4KiDSrImA9FYYvMNDECgGXLDchbQOpPbFYOlbSBTU3eliq32aIS/6rDHjUW3Cxuy7I0HgUWCptnqzjmUeZbQANNDnwkMfARWc1CCZW1oMYsM5LQ5K6fNV8hp04llJ5mz++aeaDYlM4I4N4gz7oLg4IcXUQ1mSKderRuPei4K4l+QjUDVOWih65bheRuBcywyYFE09giTXA/AIwPORv5F6yZnPziAWz4ke2m9VwC02LIYTpDiSzjEdkADrKybyLTfMIO30667s1tm0D+tj3uv37ZY+R3Z8hK5z2vBCfMHvPpwcm/qLov2WdFNHawsIJjJJpl/LaA7AMANefl2ly1GFF/pbMB3y9bgZ6wN7dHXq4YagSVwKvo2z/ZZY/SORS7c6C4LENW8DLixR4UAsbF7QeBot4FM5dab+QCH8kCaLFryOTTc3Wc6RvQ+6g25j3bZyiaFQFrYHI0CR3uXzuCjfo7Y4XCahVrXXjMxyIhMPkcueHEPwHtpbCjntjBRHAcrALNm6JfUUQ7eqMeMSQS5uQ1wq9MpWCkSnOU4GVejW+4A93ftsYEH2UWs+QuriZl/E85m8+seXtM+vUmvC57omfD8uNCqinmGOwTrrlWOGiWNUXg7WiDjhuoA7l6KTOx3Ss208kL+1XA0cbY0747Fx1YLkngHngrWPh6+Bv9ZV+jz/ZYdSFACkhYSwe4YQOFVTQCAiQNngZqgqifGYzO9RtXe0cDYzwhImU5oNpOpHQHhi5YggI2ImeK3TktHBsLAoPbJRdfvyV1kgYsmIybw7kLJVxjjLtYBjDocG9PJCEgbjYA/Qe/MrH8ueQGx2ZK7GNnm555//Mnl8C1GWroLUlAukSbeVAoUsrtgQrmCXAy6KDZOyML4jVf9XZDG6EaJV+miP9W14rfANYIUGw5xF+YTrEkEhGXo/8/i/2IgEJPZ7wUiqlFBXUX1hiMF8upV/qie63ieRr9UYEXS5vmgI5cM36t558HV1YuILBowGcLBMOagIex5Ec2N7SsOnXHEMX4IDrz3ZR+TpwAnQihbQtz18R9/6/ksJlABkIIf2mkIkCFKiMuNB/5/Iat0HRG2hiCSnts1ukHXeoAgDgQ2+PTZPx3mYhDu+i36XR97CuDx8pDAxUgdho7YkZ+PhwPbZDodC0i9oCg/cGSyjWY3ysRhlAWIQby9rs/ga28tinhPC4EkAe3YOF63OxkJjLE5pjyD5RMw8fs9fwhZRgtj6Zqwss5PVvPcbr0Q40ND5/BRPZjPFL27fh+3lh9aJFkPey5TIz7AVgzINg+A+xPQvbZ2s8AsGOl7NcUcgQPRJaibrwstKWsg8eV1YJ+DsBBgLVkcllk7/OTDj8Z5wBztYD76v8il5AOLOyhTXrRHP3KCO8Aw7YAhmLo9VbQuNNUpi/xZWFbIwdTgfAA3WSmSgJp5iNVM87k13H9MM2ADTqWwHqLsVFx+lMifR1cZEBj0LqZSIsmjyHo0YIuWl3R4TOYgYvhTj62De8ODQ85MhGDCdeWJkHM6KkzKwzOelznpIchOIfLyJ/0vot42Lym/H2WgQzNBbtD7SZ3HHSa4mOMwP4c+J7K0ieAg1isMZq5ghLwr4e47VZRFlgDYlhnw7BAC+ujTffaEUPEUQ+dvvH0IJjCTRiuhWqwedCrgDDZw1rQnTtIhBLCcyU5yl+i9IagJWHJwADHJSeI2QXFzSI5rDp67RF9YHQtv6XJiEiRvUpR1A1E4fiRJRCpRO/7J3ACY2Bw3NKiKW9hRHrX3MCiBuXOcnywU/HdI9nF9+ybXPJdo6a8CSr0S4zGNWQwtK8oDu2ATRauNXI2Md2e9141CcxVRbgtsaVp7JAbtWJ9MMCfA/F5UYuayUWJ8Y+Y2F7irz/Z9d/pC6GigOmmpaxPMJjs6POuIKaOCfGtFMYRwf1dp+VistYJgJ1iu/JvBlkGBYrxAfX0oJ2Ev+yXa+MweeVQWVpSujOgMluyf0ogJhwbf+gv0U3NdeDTE594b07saWjFyazzwU+pfK/p/D8fKOdHr67kv+0FRiSSFaMEeyQAGiEXAe+/mmlxrDWy1V/GDrmp0mOfAcqo51hAjoSzq1A1aVcGn2lcdgZw9mQ9b0E+ZvcclhkWYZwP2c8SChB9+ev/z+18Cywh1H0/hWtrN3sxQ6YUFUD04HTyK1ckiwIyA3w9ymYEmCPhJV9OWyULnnZXqkrRdLfa2FgvHObFeLu/vEExgXIFeG/vHd2Sws+KYNc98R3OsSVX+elQ5th0LOK1sk+k0dL/GvNq5k74NJ3EfNA7NH0VtVfmbLjpLVZMVjtuZ7s0AyKj1su4Z5PSHS98NDdxgQMAb6akiASU9LyJe4WtcB2PLIKZCU4GLWhoF+X95iukq+uTi1iDwoRWZrVzRfXKR8Zndou5uwlhCkjx27MgBtP1JbBBP7oRLvU2Zp7oKLC+UbjekfdpFDOPSwMstXbsJIuzBUePlYAK6L7oETjDtA5TGVS+ZfuBMZEyXXNmDjptS9tutS1kT+zrqRWR2cckr+Vu7LWMPzGECOF2DCIrNB/Pdj1xwke4WrHe2I0SGA9xb1Djhhi72goEmpWtCG9v+jjj/v8vNxXxeBlTrBWM9FsAdLF052cHB/rEDYcjmiC/edgy3jRCmqFHN+Zw9DPQ/YqSLnQwY3OB1u7wj82c5LoTaibcCeFuHR4YN3ZPWto1QHm+xxa9l/sroEqbE7EqjeyEFXyNyRDAdkS+b9GIZuzvZwTFYoOwTEA/qRTQqbZeOovOwv1ZAhFhLk3RYvRiP6FoG9sVQ037Jkw452TTcdi8RdXHXH2hpAsi7NvkuMBwbYKqaYwAr0jhVlRYpLSV7IuzjwyzGtDvFzRFeJfP7HJO5tpVGQC1tlgCDUmIF8Kn0nj/F0SClYI21+dlItQVlxBtHuJGMbNe4ax17nmwSRxhcYiKpp+qRIBcV3J1KGvA7aJwNb2CZOx6yIS9sQ41i0AHDM+6+tatLAGED6VQPzrH3AKyCxSG1lR9Dsd3bDSf7fBENl8XDdr8YpbyjjFE91HD7rKYgRQigc7zB67Acd8NOgP9k4JnMVAIKqqQbtAqLb9B5BF8QXL+qG2lJAd7v9yjT6ToRENB2Q/KJnLdiauTevsjvR02DUMBM/QOuwugynDQH4ESLZS+Vvurn8U/mthdfXfocAHk9SJiAX03GgPtfgC44bFVGYMM/nw4Z5O47LAYdPTIP1CeI8hD783vuwRGzwQD6HAHqwRzbxXQ3MzplSAq8FZoaMZuavbNFH3C+yefCOibBfyxYR9ZeVZasSGPLutuaUyyq5QW0g7+YYuwM0p2tdOMzylaWuuxjX/XCpeiiq3hlJac3SdHkXdWUUeNu2C0XYiTvmjIqdcNlVNhoFVEhGCzbSD56+Cfghn/+8d0/xe///b+sbz/8xQpZqnILMizJXLO0aeW442Rwz4uyAfTMDXkHvV5Kftaez6nrT597BWAmvS4ehVwArfo6Fi89KcFlmBqv1Vbg25HdNfvf8Oz4Ew97ScNerhazV+lnt3/rSfMkUdvmZruEmWWLYGnanWC0OIn3E5XsIBBq4ysZuISJq7EIeWDnePqpk0f+3WLUXcMJ5He07hkh7J6u0XM/eCXb/TgZf4DwtkEbA4bEMakDgY43z8EmP9Y7JtSd5B50D8qfnJ90UmLRkaKLsBr3qhqsbwObvmmwo6D8RlUdD5iJNwTjIDjs33+LM512GUqbdwzcrJBqEaFbuM3LNQgMd8fXPTXWZTjIk5JBtvs0dlSiIDNmaeBlTLdY8iD9tiD07Wtk29fFaH63d1qyL/u47jpySO/dGhb8OGTZdlgbZp5m2NiKsKE5PDe7suCMLqpfkGE4x43KcvAqNljrlMAyVFwLZ4frPhvSn2a/jvs1aN/ZSQdfA8gBgaiaw/nEM1hI/VZXLLj5Jro9FbiWCpeBnW6Bc4bmGNjUyMVICzQgW3nenxhPzXdlBL/LfQikMpkviUYhX197n05c00Xb8LmX0etdlYm29b+kFlrMz2mGnkpjPWzmhKcJArCtdqmtbQkC860fQUmj+ALp2rNLc3up7vnsujiAxG1vZx7mz7mVhwe3enJLSW/xKfa1pLpHAA1z5vT9iFeinD3kugFopeFvvvE7uAesdu9B3pdFyUl9BqaAh6Nbf7lYfLsKVY2gPAuUccMxF49T/fs335DDDRvm74DJN4u2XmP3m7ug61YXC/LLNEuy5tldzMLP7eqTCP/y6nLlk0Rt0ThBDeRPiuUQeNMcgDO6K1P4tgEPetnuBovaBMQUKSBFi95t0A2BZiXFTaZN2CwvH4Ndtt1JgdY3s9lsOn894wocq+3Va5/gqQQWlBwVFrWTJ2Yq3KF3CQAXCBCtPbbdgOf3aOdLEAr1+BwoLmer+2j21eRbY5KZmQLmEqAV4gwK/C5X2wgNIu79wNeO5AvgIKq/hAmrtxGCuA6/e21qSctik6XkDLbMluTZgbBR29WU9hjMglt/UJmDWyXqwDBW5AbS2yh8/ZolegSXbpypmKHCm0cNdANlatdFKtT7eKw5eMUHLDSfBfDtkE3nXD95yvr2HhGVEs4FoQCDB7iZFQIAgzJoNalTrfbxIxxp+QjMsmmCCjcV5KRngor/SzQ6BMFjlrYuv6hh6phQx3Q6v8PCLuq5oSZGKUK4S2qe5GPKFkuoxnuY09cRbUpAVGdBVOdBIDdwERGEkLS3YE4MuT5pHUkl/hXrzsFrgEVq4MjnEpyLdQkOnlzSmTTuowoe11HvcJlmd+scSaOuH9XN+vpxzRkGFWhADfWgH+B/awx7NlzoTerP0+paqxu9vtZrfzo3/FtVHeeyRhtRY9cwg+av2edHIaEV/begv/7cMatU2IB5jEC30z5wsz64XBtfzJc1ZRbKVf0HMmvOr8dMU2mKaLPUaUD11BkXcYIwFirPftNCLwxD85wTgeiirY9Znh6l0h1Yt4k8QhjLI7k6knbm33j43bud4Dq+1K9Xp/Z0NghhpdowIJSCTy4zrFUvR9/Z2LvIpzEQIE7XG+gDamhl3fW7xIWDAWufqgXZzmKFH34JXFqvCxl4DmapYZAF8QCxaEbpsVw/gKvHqV0aSsCnqOng1MR24lJ14y749PGzbzQKv/KKVVWp55pi5IBMXi3yKsWb/SjujrhTbAF39UoCV1YFsFhEnuq/iNqr6npMYt6DD/MswaOpIKesVNG4djEOKrLfdAU6yxvLMIM4DhTeq3lXXc7KSW5JC4cKT2l7fdSN/PN1pkj7V/qBoiWG9YZyvzooOLyUheFd0urTAu0BFSEOMi+kLoO18W6totRVYLfRfrtsEVFyqRbrldQuqFFcFuvJeQWJdqBf22uX7lsnb+Hb2QvesRTDm8+ru6LcY5JR1HcPn64iWCvQ2+bxCiip2/C15BkHZaur6St/UJJuVXR7S1lt1d1/nNayrmiL0cXCcfsfKRWmgoCf8IYD+XrDX9v68fnMcugx2m4UJnzRDwId7exUvuHnl1JjYB5PHZv2DQW9coIZWiU7R6dbK+huqmzfE9chiYg+xlIMKLRMWGqoVr9tJl3aWvC35qx6LHKD647SB3V4csNWrgXi37PJ70kXglhdnWRXzDbGH0YgV3L+jcoQ4AiLhkjipFmdqCq1BIc8RWQgz/ZpWBMwY5rPq8DydBJxa8Zo49vl/E+Lp+Wr1ZJV8KonsGJeMJdQk7tpccDfwfRiiVogDoScPliMROpE3BrO9UhPPCjxkuGDipLGFfqR2CEcCnufGRdKJJ8RHfRO7MMwukCNtZ7olPViwOZEq26x1XI9GWOY0WmjI8fVGNHTOlnMcvQwNK8HRpCQaq1X/pdwpHTXl1EZb5aqr1c9lr5UGNw3E/khqg7R2eiHm7BUyV24H9Cy26VE4vUoKiGYbKsS+Mu8/3Y/9x/A5IcAFuKgmOnoBxIiI/qWoqBq2Nv5bHZNDihE8XOvOixxi6tpLh/I9TJfSPrNkIFDKTFCU5vyNMr2RpnRm0ZlRfKdXTD2PpDC7AJEipvQmeKG9Zm3FuS7EWIRokyeGn+tDoywca4YUfp7BtZeFXzr95Dpx8gjNJZnXLqVc+9MXlFou67N0PNu3cp/++bcY5HeP1RG9LZ/WF2Ny1AP+PEYuJ9BgSvNBIdYik7SC4ksKYz7mR+2byDErZ20pLU1ZnCwbJdZbdpk+3WlQ+c91rFNDlgZ1hV6bo+Yemg0+Hs7VWuLb0NLWZO7YR7iENW6JIMV3ICUYy4DUJaKbjGueG0XsbMQQGQ/xWl8swEfmBNvQzsS4OnXOA+iAFrdXdB/gYvZaCwX1CoF/PZ8J7igoXiS9J0SHu4C/5597OMYgRUappnaYpFxlnAV+x3+hII6NiXIV5Y4Sa6yPb2a3ZUVnGuSHCuVPAcOkxcJV1Odc/JMd5P92sHQHXj+WN78R+7l+hdudMmIBXGjF29jN07e+C3VSK0Da1zwNe/MFYbtd37xwgkQPb3NGbl9Gr3dEb8DYIyXT12+pBqLdOSS1b7u6Spx2tKOXkoX+keTuUwYIxVWPTjdAvstucwIAsR9dkhlDoIZQ6euid/k/UObwx8p+5IpMpR/taO+a8v0nawxdw4K6LgDwedVXOtoA7m6OR+Snr7GkAvdDf9Sg6my3LifWkb8zMXIvAIZi/q45udjH979Szz/xh2H4e6zJ3q7w/PD19de+Odrm7hc4nQjddcXQCN71lQeWDjLdmNBu4LNpX/w4UGfAIy7374ryLP1/vAK/FKNU9bUdvvd/OP4XBjqjrw6IGzQv8Y4j05sKYrRQoPzIxJyYO9ymEJZDUKy0zMkd/wTTe6Jk3Wg4s9JOvEVgxxW23PvjfSuLp1Ue/gszIxeDcrI5rYuhdAJkskj9JVPYCRv+JJVXrF6IwH7hWsl6zG2LrbNjizZt2EfTC8nYQMewhgpTu5jjZwwos2kerlnzT7xVHngGdi1JrGhAdhNqzZlYA3RYJLHgT8EIJ5cl2w7RWbszq6DYSYymQK7MImRcxe2BYJTPF9m99n/w7KJr+foVRA9Q7ZeCZ0Swp003Mdb/92ie94Ixax/Y9au54Rb7rvN0xB5CdJahPEXGW3QEugnpiKl4SJLBkd8n7ZcMRpR1xYDmacCZ3W7WLERfmZvkfmVMeuleblJ3CH8cQiTlDtE5mekwu+rLeH4Ab9VXgq2usrolwaiOE7LJI79O+CLCLzNNI2RQ2hgjSObCHh2D1FBistbnrT1gAdm0Eye5rlc0kfVzdSuZH3PnUzY4rgBKS681h9AHZ9VYtXnuSnda6DhNPM7PEaYGAS52IH8ulf03ezMmpxeujj3zYW1MXa7ODl8/e2ZlfGpk4XuhUXsh9TgSmcJprr6z6VX7YLy0vo8tPaufsJX+cMtoNFAliUfwKWnhrAMenyBiFfPCxjyBU4FRjtZltjl8uka9xUgvDCiDLr4FIOy7DGKQR1/W2xsaalxAEtMD+kQNOoXoKl2T1bmzCngPLJwWxDJi+Mv2rTn8WYWfDcLbuez1ZDXT+GwaRhjF7ndgcE1CCqRFGfW3eMI/BaKsEb8PMC1X03VCMY8r6bBUmBCn0HEAoHAv60gvMuN/Vwrt9nXJTIV3+hZ6pUarVdvw5db1D98PmVVIA8QYuJbJXoXIjya20VtXWVdV5FH8OJPNNKcH8c6d19ThMdJna6a6GwVEQ08KSKiVruG6PfUaVoH+qXXDW1JERMb64ouF40GFzGQGiRmH7pibOuLeoRsy4eoVb70Tej/V/nQGK+gUurfgFnbWAjprNLejjgjjEPJxDjgWUF8nstooAC3UhODUVzJaP0WnwA2P8ZH5Tyw5Zj8gzgGdOIYTXsc48+XoIm/+j8MnFZZHlMAAA=="
  },
  "drum_kit_reference.py": {
    "sha256": "636b507f14605d8685b818fdf226bde160c541bf146c1e0961df21a0c964bab8",
    "gzip_base64": "H4sIAAAAAAAC/61a62/kthH/vn+F4n6QdNVq1740RddRgGviAwIUzfV6SYEaC4GWqDW7ekWUbK8d/++d4UMi9bAv1+4He0UOh8N5/obas7Ozn0rqJFXZNlWe09RJm65YH1nrNDSjDS0TnC1q0jBelZfOkdLaqRkpK4eUqVM3lNPW6crklpQHmoZnZ2crVtRV0zqkOcAyTvXzf4DBKmuqwqlJe5uzG0dNfIBHTcRvu5bl/VN3UzdVQjnXI4+szlhOV/0mCY1RevrQxsltVx65Q7gjv2kaPFIMfG4ozuHTarVKaeYkJLmlqVznZVWe0iaoc1IGrEzpg79bOfCBhQc4JY9Q/jCvSMo9Rbxx9WSIk64fNpSkMQrj+b5Y3XRlpKk18TVvG09uscclvMrvqCfpWeaUVYvLQsbjhuakZXc0bivNJHOFuPGTYLDbvk2fXX/XEMap8wvJO3rVNFXjud+Ls0lNCIU7lCekphx5u3Iz3tIikroK72jDspPSBZBIReCfa7kjd/fXYs+9Ohltu6YUPAJTNbB24w4qn9WMVD9pkls4nDaAegyqrp1aQZ4+gjnQwA3hNGcljaequJSEYXFMWaN02pD7SKmNbyZLYpSVh/fkzhXUBW1JSloCS3jLWtSiWtRQdCe+MRbL04l1vU8M/PzwvmEtjW9OLeX6fEITHsikPURwNb3LItTi+P61e6AlbcAhqlLJ4u4FC2VCuRkysmQx7RDIdZYF5xzADIZ5H1CclClrkhzJgXpoO94lGLG94RqnJAUNsi7PHVY6156UCnJMnFGa3sDSEMLaDd6TnFM/MOYhNrq8lbOfmo76e8lURYpgiWkIQ0Ztu8NswMqO9oT3DJxf5Y3w36x+D/9R0I0Qy713A0xwGJig2Kgn/PFD/MPV+7+9+3T1g49543HYWp9Lh6c4ljukSaVrt0+hYAJ6x+h9WNRvYVylW3vU4o6fnqxgD2MGcGgGXkE3EwPPsLGI+SKryTYvsYJUAs7T//+MzQfn3Wy32/idjLuYgwtsN8rQgst+N2EDthYW0wr3MTmimTx/9yj93poPesLViA26zJT/1Ji96UQg9ybTT7ZKJ6OgSjE2cxT8vCKyKk7CnajXVFWLObGG0HojgjHGdB79vSpxEeotVhlDjvGqaxL1vYUiTNtoG/Tnic4D+kCTrqWRiDfbZvShpkmLKVn5Hr8lklM/Mzi1nlORbggaYUkXkhv1LRCDkmCm7Hn90RzGHcE3ijz7gP3MXMG7rSoYIiUcgkHmLg8OcnTAtMSRbBwIarffUOkm6g80w/RjD4IwzQwRrpKmU3S8dQBXpCwThK1iL0+pqhCasWx5hBksEMLF1VFq3yoBT1BwSNtxdwe4guJO5QHcSsopE7O7k09mblET/YBtURdyIwWOCPN6kjWYN2UYi9Q8VQjKFRGyhocDkA8xGzhX/wicI0uOKG0OsKuAM6FydTF2Kqggwq9D93mhLEm1bKap0ixLbXOysrzlF5jp0QPGyXgoV7FGNFhMYxwaPMtOBypQFLlRXOWEWQTthcoiEOGBipNoEUpKWnv9YDtkMTwpP1jk1VMO7Cg40W6c5KT8Olhm3Pqnhh1YSXKF5IGwob92rAFvENWyvaXTkBmpTUSz/D6O5hcrrx3Rc6VVnSKnpfcYYpHOIWQANn4V4RD0G/awj17Iu8JjIW4Uc/ZIRUZnmMofQ1aCDiXpdxd/+ubN+fbi6zdvLuZT81RXP5Z3JGcpboIejtxTrRx1iJGClPykPJkifXe+3eq9F8SbTUBio4IWN7Rx2qpycnSomR21BSGNoLPIwHr+7WmCfMeodxDmWiWX3s/2Yv0iDDYg8ItM5rQjuhwlMlRz6PQmhv1cC/0MzwIHCflEidKwUboggG3VCP3r3S9cpHJ1iEtHQ1B0dAwEVvIuy1jChmRuflC1JlZ/lCjdULk/6wwqyUDJBBfUYfNVJLF1pcIxltMAG6aH/CAiVdYZ7og2GqOUE2igRNluWDEv8ky2GnVejxpfLGasz8haL/KcyVyDUkDlLZgPpI+rTCRr0Ixso1wMO6kkedXg7r+KcCb88PHqn1efbBeZau1KoRahK61m4Szq6sK1BJqDP2bVkSjAMuWgXRB6bv1rIn4vSaemxSjAmrSWmLxv5rcLMlvA7BWpLVuagltMXhN9QEefLfy5LbzS07ULpkDIhg2mC9CAgOn3UJJbzwA6pEtZ5frLMGBZuSTHMD05HadcwEHF8xLEErDR8o+u7AlS47pEbCGOEY164iHXXZpobUwm5/araURxioiMRwUrvbfbMPDkPthUNjSmZeru19YQIEVs//2NsutH6FMtzU54f3u+fRHgIluH30IoY6G5BeR8vnXUWkMBeZVgosLtI29OpF5OVkL4zwr6R++LzreenMnfXPSCiazAwYcAa5C2BQwlG5BZAmjVqpLBSaAwxzmAXTsiAg1UR22gPLahgmC2RE0+aSeRZTQ5QXAgrISerCAPcU/1zXZr2FJdhVZdmSKYQMDEs/F02RX1CafKup8S8QJ4mGeySM0fyvIajKuyhnqcsRI7VMHCD0meewJmwRxI6sE/csPVrP/tOV1//aJvQcRyEF02DEwCKmNjeekbJlV9Ek29UbGGjmHov4eFCDp4kwQpdGHyamnct89fweAl06ijn7mS8Uft+3AbHUJS8K7dLCtqii3auqx4mzK881kjN4oKEOP4h6HjNFr9IK8PgySDqbsqh7oXbcM/uy97krtOdgQW5OwGJMsBIyHjGzF2/peLo7UDaMPfB4DYkqPoOY3sJUp12NV4S+HJbjMS2AUaQTcYcIjAJwqOLIEUW+K+4rW00AsXK2Uw6l5fWGVXqmChj6IPCW3qdp7Fgt8H0/Q7Dc+F/bA/lhlgWDuX9qyst5g3tLotfmYheYHrLCedSXpm3hw3mXZf2Mdf3qioUpoPGctmYk+Cp0isFdlATqPReHKtHamZwXG/8CpBLNW3chI3qs3UvZrU15BlNK1O9ZIZwJMZC40jwEjy1qKxMWDdCNNEC+VGX9Sp/zZy0qIqmyFwssIWkXIPr5ZIXsVPehd1UyAs6whuU8yndptWFCX/Tr1wUG8ONFGVp0FJ76MZLBgYpxxGh4qubt0gJUdPR1ENjlgHsJMErgOS9H/DIdjEGELhLCIBN48Ahi06NfpsnsnYd/3kQkJX0DSYgNVncW2Au8DpMnZAowj2+vEVA2Tuz6XG5o7WgLn/znniCNlSzxj0n93Xcr4OA8j6JGk7yEviwBTAX2yqdYb5UqXqbWVnc8OCwzCYlsNusfCqWOYL4VWjgvUHB5t7+gAy5ifpZH2IckfekoJmpHNm4qpXJgRyk/cO6eB1/IAZ0F2FdTR2kr0KaBbftAxA1xz1Jhliozff6MIoY3YOsIwHY/1+wIBA/xdsMd1gdBn8e2DE4vunBXDx+3P0pfHaEK8KxmBlnC2wytet81fC6ZX4ClkVMa9Qye4Vj88IQEtwd0EcZe5Te6qpJ578MI7x3imOnyGexJAZQf9D8cG77NEhjXt/40z9eYCc85WVEOR7VHSx/LT7AmFWK8hZ+oRR5MZxAU1HHLvqBw6R/qVG+K45iHuYD/jUeCnlScOEWFEcp1USx1JwgP9pGhNF7bnrNVZTN0CViovhIKUZ6XL1IigW7+pg8XBPHKqXItfn6ucE6lZZci46TEn5CXFd3nH8BcShqbra6y85B1eRCycCIdYwBPIvF+jkXeFavGIeqBdOKXlYbKdEYA9TEzMiT9eoCimSMWobciV2wS0scf1LAmoRBsI1XF2y1w2DteIqMoXmj3v6hSEJBbQhof4hhXxfSEJ1E2e9MSShGgAF6DeHJFSwSOMPEmoEIn6TUbbRhe+v/gsD+2wqPCQAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_kit_reference_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
modules = {}
for name in ("electric_full_song", "drum_probe", "ace_context_chunks", "drum_kit_reference"):
    spec = importlib.util.spec_from_file_location(name, TOOLS / (name + ".py"))
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    sys.modules[name] = module; modules[name] = module
context = modules["ace_context_chunks"]; kit = modules["drum_kit_reference"]
ROOT = modules["electric_full_song"].discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = modules["electric_full_song"].discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source/runtime bridge belum ditemukan. Pakai VM aktif atau setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    modules["electric_full_song"].model_python(ROOT)
EXPECTED_CONTROL_SHA = "52486d77017b93834de24aee63a6c5ca31a6b8d7a8c1fcc682ba0eae54f3f649"
EXPECTED_REFERENCE_SHA = "172e1bb34cc3202c44020e9cd2501412d5f74caffe6c6d925c8e5f198a79fa64"
KIT_PLAN = None
KIT_RESULT_ZIP = globals().get("KIT_RESULT_ZIP")
if not USE_SAVED_ZIP:
    hint = globals().get("DRUM_CHUNK_PLAN")
    candidates = ([Path(hint)] if hint and Path(hint).is_file() else [])
    candidates += sorted(ROOT.glob("runs/drum_chunks90_*/plan.json"), key=lambda p:p.stat().st_mtime, reverse=True)
    for path in dict.fromkeys(candidates):
        try:
            folder, plan, _ = context.load_plan(path)
            control, _ = kit.cached_chunk(folder, plan, 0)
            reference, _ = kit.cached_chunk(folder, plan, 1)
            if context.sha256(control) == EXPECTED_CONTROL_SHA and context.sha256(reference) == EXPECTED_REFERENCE_SHA:
                KIT_PLAN = path; break
        except (OSError, ValueError, KeyError):
            continue
SOURCE_PATH = globals().get("SOURCE_PATH")
if KIT_PLAN is None:
    if KIT_RESULT_ZIP is None or not Path(KIT_RESULT_ZIP).is_file():
        print("Upload drum_chunked_result.zip; tambah piano asli jika SOURCE_PATH sudah tidak tersedia")
        uploaded = files.upload()
        archives = [n for n in uploaded if Path(n).suffix.lower() == ".zip"]
        audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
        assert len(archives) == 1, "Pilih drum_chunked_result.zip yang berisi WAV chunk"
        KIT_RESULT_ZIP = TOOLS / "drum_chunked_result.zip"
        KIT_RESULT_ZIP.write_bytes(uploaded[archives[0]])
        if audios:
            assert len(audios) == 1, "Pilih satu piano asli"
            SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
            SOURCE_PATH.write_bytes(uploaded[audios[0]])
    if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
        uploaded = files.upload()
        audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
        assert len(audios) == 1, "Pilih satu piano asli"
        SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
        SOURCE_PATH.write_bytes(uploaded[audios[0]])
print("Runtime dipakai kembali:", ROOT)
print("Kontrol WAV cached:" if KIT_PLAN else "Kontrol WAV dari result ZIP:", KIT_PLAN or KIT_RESULT_ZIP)


## Jalankan satu pembanding

Parameter aktual dan hash input piano dibandingkan dengan kontrol. Jika ada perubahan selain jalur file source yang isinya identik dan reference_audio, hasil ditandai gagal. Tidak ada EQ, retiming, atau kick replacement. Evaluasi kit yang lebih ringan sekaligus apakah kick/groove ikut berubah.


In [ ]:
from IPython.display import Audio, display
KIT_OUT = ROOT / "runs" / ("drum_kit_ref1_on0_" + uuid.uuid4().hex[:10])
comparison = kit.compare(ROOT, KIT_OUT, plan_path=KIT_PLAN,
    result_archive=None if KIT_PLAN else KIT_RESULT_ZIP, source=SOURCE_PATH,
    target=0, reference=1, execute=not DRY_RUN,
    expected_control_sha=EXPECTED_CONTROL_SHA, expected_reference_sha=EXPECTED_REFERENCE_SHA)
print("Status:", comparison["status"], "|", KIT_OUT)
if not DRY_RUN:
    for label, path in [("Referensi kit chunk 1", "reference_preview.mp3"),
                        ("Kontrol chunk 0 + piano", "control_mix_preview.mp3"),
                        ("Chunk 0 dengan referensi + piano", "candidate/mix_preview.mp3")]:
        print(label)
        display(Audio(filename=str(KIT_OUT / path), normalize=False))


## Simpan hasil dan kirim feedback

In [ ]:
for name in ("drum_kit_result.zip", "drum_kit_feedback.zip"):
    path = KIT_OUT / name
    if path.is_file():
        files.download(str(path))
if DRY_RUN:
    print("Plan saja; tidak ada audio candidate dari model")
